# Reconstrucción v2
Nueva ejecución de la receta auditada; no se reutilizan salidas ni run_id de v1. Misma entrada congelada y folds. La selección de modelos incluirá seis candidatos y el criterio del negocio desde el diseño. 2014J es un benchmark previamente observado.

# 02 · Features y preparación sin fuga

Dos derivadas motivadas por EDA: log1p de clics actuales e intensidad por día activo. Se implementan en src/pipeline.py, que también centraliza imputación, codificación y escalado. Se analiza solo desarrollo día 56.

Esta etapa verifica las transformaciones; no entrena clasificadores. La evidencia de aporte predictivo por ablación se obtendrá **después de las dos líneas base**, sin abrir hold-out.

In [1]:
from pathlib import Path
import sys,json,hashlib,subprocess
here = Path.cwd().resolve()
BASE = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if BASE is None and (here/'pset_3/config/protocolo-v2.json').is_file(): BASE=here/'pset_3'
assert BASE is not None, 'Ejecutar desde pset_3 o notebooks; revisar README'
sys.path.insert(0,str(BASE))
from IPython.display import display,Markdown
from src.features_evidencia import validar
from src.pipeline import construir_pipeline,construir_preparacion,FeaturesAlCorte
from src.tracking import configurar
from src.common import sha256
summary,folds,descriptivos,contrato=validar()
display(__import__('pandas').DataFrame(contrato['features']))
print('Snapshot:',summary['snapshot_id'])
print('Split:',summary['split_id'])

,name,formula,source_columns,evidence
0,log1p_clicks_current_14d,log(1 + clicks_current_14d),[clicks_current_14d],"EDA: mediana 53, máximo 7028, asimetría 8.82. ..."
1,clicks_per_active_day_14d,clicks_current_14d / active_days_current_14d s...,"[clicks_current_14d, active_days_current_14d]",EDA: actividad y resultado asociados; normaliz...


Snapshot: 0a75615bd8ff6b166d366cf78c8303845852cbeb23387ac24a64fe4962f01027
Split: 231cd638f375de2a8c70b4482fdf6a98dbe022371a409f1fc92bc97189cf7736


## 1. Evidencia, fórmulas y disponibilidad

La cola de clics (mediana 53, máximo 7028) motiva log1p, definido también en cero. Es monótona y no agrega información nueva a los clics: puede cambiar la representación útil para modelos lineales/redes, sin prometer mejora en árboles.

La intensidad combina clics y días activos observados en la misma ventana de 14 días. Si no hay días activos, se conserva NaN; el Pipeline imputa y agrega indicador con estadísticas de entrenamiento. No se confunde intensidad desconocida con cero. Las medianas por clase son evidencia descriptiva de desarrollo, no prueba de aporte ni resultados de un clasificador.

Ambas usan únicamente columnas agregadas por Spark al corte. La protección temporal hereda las pruebas de perturbación del PSet 2; aquí se comprueba la invariancia frente a columnas futuras ajenas al contrato y se rechaza un último evento posterior al corte. Esto no certifica la fecha institucional de disponibilidad de todos los atributos.

In [2]:
display(descriptivos)
print('Casos sin días activos actuales:',summary['zero_current_active_days'])
assert summary['zero_current_active_days']==summary['derived_missing']
print('No se eliminaron observaciones ni se reemplazó indefinido por cero.')

,feature,clase,observados,faltantes,mediana
0,log1p_clicks_current_14d,Approved,9219,0,4.356709
1,log1p_clicks_current_14d,Fail,4918,0,2.995732
2,log1p_clicks_current_14d,Withdrawn,2194,0,3.610918
3,clicks_per_active_day_14d,Approved,8716,503,15.125000
4,clicks_per_active_day_14d,Fail,3300,1618,13.354167
5,clicks_per_active_day_14d,Withdrawn,1713,481,13.500000


Casos sin días activos actuales: 2602
No se eliminaron observaciones ni se reemplazó indefinido por cero.


## 2. Preparación aprendida exclusivamente en cada fold

FeaturesAlCorte selecciona por nombres y descarta etiquetas, IDs, folds y auditoría como entradas del modelo. Convierte pd.NA a np.nan. Numéricas: mediana e indicadores de faltantes, luego StandardScaler. Una columna completamente ausente en fit se conserva con fallback cero e indicador. Categóricas: nivel fijo __MISSING__, one-hot y categorías desconocidas ignoradas. El nivel ausente no tiene orden numérico.

Las flags is_imd_missing y has_recorded_activity son explícitas; recencia ausente no se convierte en recencia cero antes del Pipeline. No se recortan extremos. Un pipeline estático exige solo entradas estáticas y nunca depende de actividad; las dos variantes longitudinales comparten bases y difieren solo en las derivadas.

Se ajustan preparadores en los cinco train-folds de cada variante. La validación transforma sin fit. Se contrastan medianas/escalado con los datos del train-fold y se exige que transform no cambie el estado aprendido.

In [3]:
display(folds)
assert len(folds)==15
assert folds[['salida_finita','medianas_solo_fit','escalado_solo_fit','transform_sin_reajuste','etiquetas_ids_futuro_ignorados']].all().all()
assert folds.alumnos_compartidos.eq(0).all()
print('15 ajustes de preparación verificados; cero clasificadores entrenados.')

,variante,fold,filas_fit,filas_validacion,columnas_salida,salida_finita,medianas_solo_fit,escalado_solo_fit,transform_sin_reajuste,etiquetas_ids_futuro_ignorados,alumnos_compartidos
0,estatica,0,13055,3276,41,True,True,True,True,True,0
1,estatica,1,13049,3282,41,True,True,True,True,True,0
2,estatica,2,13066,3265,41,True,True,True,True,True,0
3,estatica,3,13054,3277,41,True,True,True,True,True,0
4,estatica,4,13100,3231,41,True,True,True,True,True,0
5,longitudinal_original,0,13055,3276,51,True,True,True,True,True,0
6,longitudinal_original,1,13049,3282,51,True,True,True,True,True,0
7,longitudinal_original,2,13066,3265,51,True,True,True,True,True,0
8,longitudinal_original,3,13054,3277,51,True,True,True,True,True,0
9,longitudinal_original,4,13100,3231,51,True,True,True,True,True,0


15 ajustes de preparación verificados; cero clasificadores entrenados.


## 3. Comparación futura y límites

La siguiente etapa ejecutará primero DummyClassifier y la política reactiva. Después comparará candidatos y ablación con exactamente los folds guardados: estático, longitudinal sin derivadas y con derivadas. El aporte individual requiere además activar cada derivada por separado y reportar la diferencia frente a la dispersión entre folds; no se atribuye aquí una mejora a ninguna.

La selección se hará sin consultar prueba. La codificación densa se mantendrá igual para todas las familias y se medirá memoria/tiempo. No se ha cambiado el protocolo congelado de particiones; este contrato versiona la preparación posterior a EDA.

In [4]:
reference=json.loads((BASE/'docs/portabilidad/referencias-historicas.json').read_text())['features']
assert summary['snapshot_id']==reference['snapshot_id'] and summary['split_id']==reference['split_id']
assert summary['development_rows']==reference['development_rows']
summary.update({k:reference[k] for k in ['run_id','experiment_id','notebook_source_sha256'] if k in reference})
summary['replay_entrega']='Preparación recalculada; run_id histórico, sin entrenamiento de clasificadores'
(BASE/'docs/features/resumen.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+chr(10))
print('Preparación reproducida; referencia histórica:',reference['run_id'])
print('Ablación y evaluación final disponibles en 03-05. 2014J es un benchmark reutilizado.')


🏃 View run features-preparacion-dia56 at: http://127.0.0.1:5001/#/experiments/6/runs/d5c53f801c314ff9b15b63a6c2a53d12
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/6
Corrida features FINISHED: d5c53f801c314ff9b15b63a6c2a53d12
Ablación pendiente. La versión 2 no ha evaluado prueba; 2014J sí fue observado en v1.
